# Qwen2.5 7B-Suspect Reference-Comparison Experiment

This notebook is a focused fixed-budget variant of the primary estimator convergence experiment.

Protocol:

- References `A` are `14b` and `32b`.
- The canary prompts are fixed to the `7b` canaries for every comparison.
- Each graph has two curves: the reference model itself and the `7b` suspect.
- The base configuration is copied from the main primary graphing notebook: same decoding seed, same fixed samples per context, and no until-convergence allocation.


## 1. Place Files

This notebook uses the saved primary prompt-selection payloads. It will search for `logit_payloads/` from this folder, or for `primary_experiment/logit_payloads/` from the repo root.

Expected payload examples:

```text
primary_experiment/logit_payloads/qwen25_7b_8bit_prompt_1_logits.pt
primary_experiment/logit_payloads/qwen25_14b_8bit_prompt_4_logits.pt
primary_experiment/logit_payloads/qwen25_32b_4bit_prompt_6_logits.pt
```


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in PROJECT_ROOT_CANDIDATES if (path / "logit_helpers.py").exists() and (path / "estimators.py").exists()),
    None,
)
assert PROJECT_ROOT is not None, "Could not find repo root containing logit_helpers.py and estimators.py."
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PAYLOAD_DIR_CANDIDATES = [
    Path.cwd() / "logit_payloads",
    Path.cwd() / "primary_experiment" / "logit_payloads",
    PROJECT_ROOT / "primary_experiment" / "logit_payloads",
]
PAYLOAD_DIR = next((path for path in PAYLOAD_DIR_CANDIDATES if path.exists()), None)
assert PAYLOAD_DIR is not None, "Missing logit_payloads directory. Expected it in ./logit_payloads or ./primary_experiment/logit_payloads."
print("Using project root:", PROJECT_ROOT)
print("Using payload directory:", PAYLOAD_DIR)

import numpy as np
import torch

from logit_helpers import make_random_decoding_params
from logit_helpers import make_random_decoding_probability_vector
from logit_helpers import make_cumulative_probability_vector
from estimators import sample_g_for_fixed_iterations
from estimators import naive_method_i_estimator
from estimators import naive_method_ii_estimator

print("Ready. Payload files found:", len(list(PAYLOAD_DIR.glob("*.pt"))))


Using project root: C:\Users\beatn\LLM-Ownership-Identification-Draft-1
Using payload directory: C:\Users\beatn\LLM-Ownership-Identification-Draft-1\primary_experiment\logit_payloads


Ready. Payload files found: 18


## 2. Experiment Configuration

The token pair, random decoding seed, and fixed Qwen2.5-7B-Instruct canary prompt map define this focused protocol. For each reference model, we compare two suspect distributions on the same 7B canaries: the matching reference model and Qwen2.5-7B-Instruct.

The Qwen2.5-14B-Instruct reference runs use `7,500` one-token samples per context, matching the main primary graphing notebook. The Qwen2.5-32B-Instruct reference runs use `15,000` one-token samples per context. If a target token has zero probability after decoding, the corresponding run is marked undefined.


In [2]:
TOKEN_A = 15  # decoded as "0"
TOKEN_B = 16  # decoded as "1"
INTEREST_TOKENS = (TOKEN_A, TOKEN_B)

FIXED_SAMPLES_PER_CONTEXT = 7_500
FIXED_SAMPLES_PER_CONTEXT_BY_REFERENCE = {
    "14b": 7_500,
    "32b": 15_000,
}

SHARED_DECODING_SEED = 20260722
SAMPLING_SEED_BASE = 910000

MODEL_ORDER = {"7b": 0, "14b": 1, "32b": 2}

MODELS = {
    "7b": {
        "label": "Qwen2.5-7B-Instruct",
        "prefix": "qwen25_7b_8bit",
        "canaries": {1: 1, 2: 2, 4: 3},
    },
    "14b": {
        "label": "Qwen2.5-14B-Instruct",
        "prefix": "qwen25_14b_8bit",
        "canaries": {1: 1, 2: 4, 4: 5},
    },
    "32b": {
        "label": "Qwen2.5-32B-Instruct",
        "prefix": "qwen25_32b_4bit",
        "canaries": {1: 2, 2: 6, 4: 4},
    },
}

REFERENCE_MODEL_KEYS = ["14b", "32b"]
SUSPECT_MODEL_KEYS_BY_REFERENCE = {
    "14b": ["14b", "7b"],
    "32b": ["32b", "7b"],
}
CANARY_PROVIDER_KEY = "7b"


## 3. Loading and Simulation Helpers

The helper names make the canary ownership explicit: `reference_key` chooses the prompt IDs; `suspect_key` chooses the model distribution being sampled on those prompt IDs.


In [3]:
def payload_path(model_key: str, prompt_id: int) -> Path:
    prefix = MODELS[model_key]["prefix"]
    return PAYLOAD_DIR / f"{prefix}_prompt_{prompt_id}_logits.pt"


def load_logits(model_key: str, prompt_id: int) -> np.ndarray:
    path = payload_path(model_key, prompt_id)
    if not path.exists():
        raise FileNotFoundError(path)
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    return payload["logits"].detach().cpu().numpy().astype(np.float64, copy=False)


def reference_canary_prompts(reference_key: str) -> dict[int, int]:
    """Return the fixed 7B canary prompt map used for every comparison."""
    return dict(MODELS[CANARY_PROVIDER_KEY]["canaries"])


def build_reference_z(reference_key: str) -> dict[tuple[int, int], float]:
    """Build z from A logits on the fixed 7B canary prompts."""
    z = {}
    for slot, prompt_id in reference_canary_prompts(reference_key).items():
        logits = load_logits(reference_key, prompt_id)
        z[(slot, 0)] = float(logits[TOKEN_A])
        z[(slot, 1)] = float(logits[TOKEN_B])
    return z


def category_probabilities(cdf) -> dict[int, float]:
    edges = np.concatenate(([0.0], np.asarray(cdf.cumulative_probs, dtype=np.float64)))
    masses = np.diff(edges)
    return {int(category): float(mass) for category, mass in zip(cdf.categories, masses)}


def target_token_status(probs_by_category: dict[int, float]) -> str:
    missing = []
    if probs_by_category.get(TOKEN_A, 0.0) <= 0.0:
        missing.append(f"token {TOKEN_A}")
    if probs_by_category.get(TOKEN_B, 0.0) <= 0.0:
        missing.append(f"token {TOKEN_B}")
    if missing:
        return "undefined: " + ", ".join(missing) + " has zero probability after decoding"
    return "sampled"


def build_suspect_cdfs_on_reference_canaries(reference_key: str, suspect_key: str, params) -> dict[int, object]:
    """Build B sampling CDFs on the fixed 7B canary prompt IDs."""
    cdfs = {}
    for slot, prompt_id in reference_canary_prompts(reference_key).items():
        logits = load_logits(suspect_key, prompt_id)
        probs = make_random_decoding_probability_vector(logits, params)
        cdfs[slot] = make_cumulative_probability_vector(probs, interest_tokens=INTEREST_TOKENS)
    return cdfs


def final_g_dictionary(g_sequences: dict[int, list[float]]) -> dict[int, float]:
    return {
        slot: float(values[-1]) if values else np.nan
        for slot, values in g_sequences.items()
    }


def final_estimator_values(z, g_sequences) -> dict[str, float]:
    g = final_g_dictionary(g_sequences)
    return {
        "method_i_final": float(naive_method_i_estimator(z, g)),
        "method_ii_final": float(naive_method_ii_estimator(z, g)),
    }


def run_status(context_summaries: dict[int, dict]) -> str:
    issues = [
        f"C_{slot}: {summary['status']}"
        for slot, summary in context_summaries.items()
        if summary["status"] != "sampled"
    ]
    return "; ".join(issues) if issues else "sampled"


## 4. Run Fixed-Budget Trials

For each reference model, the notebook evaluates both the reference model itself and the `Qwen2.5-7B-Instruct` suspect on the fixed `Qwen2.5-7B-Instruct` canary prompts. The Qwen2.5-14B-Instruct reference uses `7,500` samples per context; the Qwen2.5-32B-Instruct reference uses `15,000` samples per context. Undefined target-token support is preserved as an experiment result.


In [4]:
def run_reference_canary_trial(reference_key: str, suspect_key: str) -> dict:
    print("=" * 80)
    print(f"Reference Model: {MODELS[reference_key]['label']} | Suspect Model: {MODELS[suspect_key]['label']}")
    print("Canary prompts supplied by fixed provider:", CANARY_PROVIDER_KEY, reference_canary_prompts(reference_key))

    decode_seed = SHARED_DECODING_SEED
    params = make_random_decoding_params(seed=decode_seed, tokens_to_bias=INTEREST_TOKENS)
    print("Decoding params:", params)

    z = build_reference_z(reference_key)
    cdfs = build_suspect_cdfs_on_reference_canaries(reference_key, suspect_key, params)
    sample_budget = FIXED_SAMPLES_PER_CONTEXT_BY_REFERENCE[reference_key]

    g_sequences = {}
    context_summaries = {}
    for slot in (1, 2, 4):
        prompt_id = reference_canary_prompts(reference_key)[slot]
        probs_by_category = category_probabilities(cdfs[slot])
        status = target_token_status(probs_by_category)
        print(f"slot C_{slot} prompt {prompt_id} category probabilities:", probs_by_category)

        if status != "sampled":
            print(f"WARNING: C_{slot} {status}. Skipping this context.")
            g_sequences[slot] = []
        else:
            sample_seed = (
                SAMPLING_SEED_BASE
                + 1000 * MODEL_ORDER[reference_key]
                + 100 * MODEL_ORDER[suspect_key]
                + slot
            )
            rng = np.random.default_rng(sample_seed)
            g_sequences[slot] = sample_g_for_fixed_iterations(
                cdfs[slot],
                num_samples=sample_budget,
                rng=rng,
                token_a=TOKEN_A,
                token_b=TOKEN_B,
            )

        final_g = float(g_sequences[slot][-1]) if g_sequences[slot] else np.nan
        context_summaries[slot] = {
            "prompt_id": prompt_id,
            "finite_g_values": len(g_sequences[slot]),
            "final_g": final_g,
            "p_token_a": probs_by_category.get(TOKEN_A, 0.0),
            "p_token_b": probs_by_category.get(TOKEN_B, 0.0),
            "p_other": probs_by_category.get(-1, 0.0),
            "status": status,
            "sample_budget": sample_budget,
        }
        print(f"slot C_{slot} finite g length:", context_summaries[slot]["finite_g_values"])
        print(f"slot C_{slot} final g:", context_summaries[slot]["final_g"])

    estimator_summary = final_estimator_values(z, g_sequences)
    print("Run status:", run_status(context_summaries))
    print("Final Naive Method I estimator:", estimator_summary["method_i_final"])
    print("Final Naive Method II estimator:", estimator_summary["method_ii_final"])
    return {
        "reference_key": reference_key,
        "suspect_key": suspect_key,
        "canary_provider_key": CANARY_PROVIDER_KEY,
        "params": params,
        "z": z,
        "cdfs": cdfs,
        "g_sequences": g_sequences,
        "context_summaries": context_summaries,
        "run_status": run_status(context_summaries),
        "estimator_summary": estimator_summary,
    }


trial_results = {}

for reference_key in REFERENCE_MODEL_KEYS:
    for suspect_key in SUSPECT_MODEL_KEYS_BY_REFERENCE[reference_key]:
        trial_results[(reference_key, suspect_key)] = run_reference_canary_trial(reference_key, suspect_key)


Reference Model: Qwen2.5-14B-Instruct | Suspect Model: Qwen2.5-14B-Instruct
Canary prompts supplied by fixed provider: 7b {1: 1, 2: 2, 4: 3}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)


slot C_1 prompt 1 category probabilities: {15: 0.5, 16: 0.5, -1: 0.0}


slot C_1 finite g length: 7500
slot C_1 final g: 0.03360316162341803
slot C_2 prompt 2 category probabilities: {15: 0.08236937877579752, 16: 0.9176306212242025, -1: 0.0}
slot C_2 finite g length: 7500
slot C_2 final g: -2.427941715681653
slot C_4 prompt 3 category probabilities: {15: 0.8974790920623604, 16: 0.10252090793763957, -1: 0.0}


slot C_4 finite g length: 7500
slot C_4 final g: 2.183961783457808
Run status: sampled
Final Naive Method I estimator: -0.014321140278916511
Final Naive Method II estimator: 0.03477089389931021
Reference Model: Qwen2.5-14B-Instruct | Suspect Model: Qwen2.5-7B-Instruct
Canary prompts supplied by fixed provider: 7b {1: 1, 2: 2, 4: 3}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)


slot C_1 prompt 1 category probabilities: {15: 0.9080498024358086, 16: 0.09195019756419143, -1: 0.0}


...

slot C_4 finite g length: 15000
slot C_4 final g: -0.967395192166423
Run status: sampled
Final Naive Method I estimator: -0.0046148379253045935
Final Naive Method II estimator: 0.009705269670467032
Reference Model: Qwen2.5-32B-Instruct | Suspect Model: Qwen2.5-7B-Instruct
Canary prompts supplied by fixed provider: 7b {1: 1, 2: 2, 4: 3}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)


slot C_1 prompt 1 category probabilities: {15: 0.9080498024358086, 16: 0.09195019756419143, -1: 0.0}


slot C_1 finite g length: 15000
slot C_1 final g: 2.25794125051738
slot C_2 prompt 2 category probabilities: {15: 0.5894245906718125, 16: 0.4105754093281875, -1: 0.0}


slot C_2 finite g length: 15000
slot C_2 final g: 0.345255860502963
slot C_4 prompt 3 category probabilities: {15: 0.8974790920623604, 16: 0.10252090793763957, -1: 0.0}


slot C_4 finite g length: 15000
slot C_4 final g: 2.1832290630952906
Run status: sampled
Final Naive Method I estimator: -0.15727183028291292
Final Naive Method II estimator: -0.05429902109720297


## 5. Context and Final Estimator Summary

The `canary_provider` column should always equal `7b` in this focused protocol. Undefined rows indicate that the fixed decoding parameters/top-p cutoff made at least one target token unsampleable for that context.


In [5]:
context_rows = []
estimator_rows = []

for (reference_key, suspect_key), result in trial_results.items():
    for slot in (1, 2, 4):
        summary = result["context_summaries"][slot]
        context_rows.append({
            "A": result["reference_key"],
            "B": suspect_key,
            "canary_provider": result["canary_provider_key"],
            "slot": slot,
            "prompt_id": summary["prompt_id"],
            "finite_g_values": summary["finite_g_values"],
            "final_g": summary["final_g"],
            "p_token_a": summary["p_token_a"],
            "p_token_b": summary["p_token_b"],
            "p_other": summary["p_other"],
            "status": summary["status"],
            "sample_budget": summary["sample_budget"],
        })

    estimator_rows.append({
        "A": result["reference_key"],
        "B": suspect_key,
        "canary_provider": result["canary_provider_key"],
        "method_i_final": result["estimator_summary"]["method_i_final"],
        "method_ii_final": result["estimator_summary"]["method_ii_final"],
        "status": result["run_status"],
        "expected": "near 0" if result["reference_key"] == suspect_key else "nonzero",
    })

print("Context stabilization rows:")
for row in context_rows:
    print(row)

print("\nFinal estimator rows:")
for row in estimator_rows:
    print(row)


Context stabilization rows:
{'A': '14b', 'B': '14b', 'canary_provider': '7b', 'slot': 1, 'prompt_id': 1, 'finite_g_values': 7500, 'final_g': 0.03360316162341803, 'p_token_a': 0.5, 'p_token_b': 0.5, 'p_other': 0.0, 'status': 'sampled', 'sample_budget': 7500}
{'A': '14b', 'B': '14b', 'canary_provider': '7b', 'slot': 2, 'prompt_id': 2, 'finite_g_values': 7500, 'final_g': -2.427941715681653, 'p_token_a': 0.08236937877579752, 'p_token_b': 0.9176306212242025, 'p_other': 0.0, 'status': 'sampled', 'sample_budget': 7500}
{'A': '14b', 'B': '14b', 'canary_provider': '7b', 'slot': 4, 'prompt_id': 3, 'finite_g_values': 7500, 'final_g': 2.183961783457808, 'p_token_a': 0.8974790920623604, 'p_token_b': 0.10252090793763957, 'p_other': 0.0, 'status': 'sampled', 'sample_budget': 7500}
{'A': '14b', 'B': '7b', 'canary_provider': '7b', 'slot': 1, 'prompt_id': 1, 'finite_g_values': 7500, 'final_g': 2.338303175596125, 'p_token_a': 0.9080498024358086, 'p_token_b': 0.09195019756419143, 'p_other': 0.0, 'status':

## 6. Estimator Convergence Graphs

These plots treat the three reference-canary context traces as parallel sequences. At parallel step `t`, the estimator uses `g_1[t]`, `g_2[t]`, and `g_4[t]`. The x-axis reports total one-token calls across the three contexts, so the plotted x-value is `3t`.


In [6]:
from IPython.display import HTML, display

SLOTS = (1, 2, 4)
METHODS = {
    "Naive Method I": naive_method_i_estimator,
    "Naive Method II": naive_method_ii_estimator,
}

COLORS = {
    "7b": "#2563eb",
    "14b": "#dc2626",
    "32b": "#16a34a",
}


def parallel_estimator_curve(
    result: dict,
    estimator_fn,
) -> tuple[np.ndarray, np.ndarray]:
    """Build an estimator trajectory from parallel context iterations.

    At iteration t, the estimator uses g_1[t], g_2[t], and g_4[t]. The x-axis
    is therefore the number of sampling iterations per context, not an
    interleaved or summed count across contexts.
    """
    g_sequences = result["g_sequences"]
    lengths = {slot: len(g_sequences[slot]) for slot in SLOTS}
    if any(lengths[slot] == 0 for slot in SLOTS):
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)

    curve_length = min(lengths.values())
    xs = np.arange(1, curve_length + 1, dtype=np.float64) * len(SLOTS)
    ys = []

    for index in range(curve_length):
        current_g = {
            slot: float(g_sequences[slot][index])
            for slot in SLOTS
        }
        ys.append(float(estimator_fn(result["z"], current_g)))

    return xs, np.asarray(ys, dtype=np.float64)


def _nice_number(value: float) -> str:
    value = float(value)
    if abs(value) >= 1000 or (abs(value) < 0.001 and value != 0.0):
        return f"{value:.2e}"
    if abs(value) >= 10:
        return f"{value:.1f}"
    return f"{value:.3f}"


def _polyline_points(xs, ys, x_min, x_max, y_min, y_max, left, top, plot_w, plot_h) -> str:
    finite = np.isfinite(xs) & np.isfinite(ys)
    xs = xs[finite]
    ys = ys[finite]
    if len(xs) == 0:
        return ""

    x_span = max(float(x_max - x_min), 1.0)
    y_span = max(float(y_max - y_min), 1e-12)
    px = left + ((xs - x_min) / x_span) * plot_w
    py = top + (1.0 - ((ys - y_min) / y_span)) * plot_h
    return " ".join(f"{x:.2f},{y:.2f}" for x, y in zip(px, py))


def model_label(model_key: str) -> str:
    return MODELS[model_key]["label"]


def legend_label_for_result(result: dict) -> str:
    suspect_key = result["suspect_key"]
    suspect_label = model_label(suspect_key)
    if result["run_status"] == "sampled":
        return suspect_label
    issue_slots = []
    for slot, summary in result["context_summaries"].items():
        if summary["status"] != "sampled":
            issue_slots.append(f"C_{slot}")
    issue_text = ",".join(issue_slots) if issue_slots else "undefined"
    return f"{suspect_label} (undefined: {issue_text})"


def render_svg_plot(reference_key: str, method_name: str, curves: dict[str, tuple[np.ndarray, np.ndarray]]) -> str:
    width = 930
    height = 440
    left = 78
    right = 165
    top = 46
    bottom = 64
    plot_w = width - left - right
    plot_h = height - top - bottom

    x_arrays = [xy[0] for xy in curves.values() if len(xy[0])]
    y_arrays = [xy[1][np.isfinite(xy[1])] for xy in curves.values() if len(xy[1]) and np.any(np.isfinite(xy[1]))]
    has_finite_data = bool(x_arrays) and bool(y_arrays)

    if has_finite_data:
        all_x = np.concatenate(x_arrays)
        all_y = np.concatenate(y_arrays)
        x_min = 0.0
        x_max = float(np.max(all_x))

        final_values = np.array([
            xy[1][np.isfinite(xy[1])][-1]
            for xy in curves.values()
            if len(xy[1]) and np.any(np.isfinite(xy[1]))
        ], dtype=np.float64)
        display_y = all_y
        if len(display_y) > 20:
            low = float(np.quantile(display_y, 0.01))
            high = float(np.quantile(display_y, 0.99))
            display_y = display_y[(display_y >= low) & (display_y <= high)]
            display_y = np.concatenate([display_y, final_values, np.array([0.0])])

        y_min = float(np.min(display_y))
        y_max = float(np.max(display_y))
        if y_min == y_max:
            y_min -= 1.0
            y_max += 1.0
        pad = 0.08 * (y_max - y_min)
        y_min -= pad
        y_max += pad
        y_min = min(y_min, 0.0)
        y_max = max(y_max, 0.0)
    else:
        x_min = 0.0
        x_max = float(FIXED_SAMPLES_PER_CONTEXT * len(SLOTS))
        y_min = -1.0
        y_max = 1.0

    def x_coord(x):
        return left + ((x - x_min) / max(x_max - x_min, 1.0)) * plot_w

    def y_coord(y):
        return top + (1.0 - ((y - y_min) / max(y_max - y_min, 1e-12))) * plot_h

    zero_y = y_coord(0.0)
    svg = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" style="max-width:100%; height:auto; font-family: Arial, sans-serif;">',
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="white"/>',
        f'<text x="{width/2:.1f}" y="24" text-anchor="middle" font-size="18" font-weight="700">Estimator Convergence for {method_name} [Reference Model: {model_label(reference_key)}]</text>',
        f'<rect x="{left}" y="{top}" width="{plot_w}" height="{plot_h}" fill="#fafafa" stroke="#d4d4d8"/>',
    ]

    if not has_finite_data:
        svg.append(f'<text x="{left + plot_w/2:.1f}" y="{top + plot_h/2:.1f}" text-anchor="middle" font-size="15" fill="#991b1b">No finite estimator curves; see undefined legend entries.</text>')

    for frac in np.linspace(0, 1, 5):
        x = left + frac * plot_w
        x_value = x_min + frac * (x_max - x_min)
        svg.append(f'<line x1="{x:.2f}" y1="{top}" x2="{x:.2f}" y2="{top+plot_h}" stroke="#e5e7eb"/>')
        svg.append(f'<text x="{x:.2f}" y="{top+plot_h+24}" text-anchor="middle" font-size="12" fill="#374151">{int(round(x_value))}</text>')

    for frac in np.linspace(0, 1, 5):
        y = top + frac * plot_h
        y_value = y_max - frac * (y_max - y_min)
        svg.append(f'<line x1="{left}" y1="{y:.2f}" x2="{left+plot_w}" y2="{y:.2f}" stroke="#e5e7eb"/>')
        svg.append(f'<text x="{left-10}" y="{y+4:.2f}" text-anchor="end" font-size="12" fill="#374151">{_nice_number(y_value)}</text>')

    final_value_rows = []
    for suspect_key, (xs, ys) in curves.items():
        color = COLORS.get(suspect_key, "#525252")
        points = _polyline_points(xs, np.clip(ys, y_min, y_max), x_min, x_max, y_min, y_max, left, top, plot_w, plot_h)
        if points:
            svg.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2"/>')
        if len(xs):
            finite = np.isfinite(ys)
            if np.any(finite):
                last_x = xs[finite][-1]
                raw_last_y = float(ys[finite][-1])
                last_y = float(np.clip(raw_last_y, y_min, y_max))
                svg.append(f'<circle cx="{x_coord(last_x):.2f}" cy="{y_coord(last_y):.2f}" r="3.5" fill="{color}"/>')
                final_value_rows.append((suspect_key, color, raw_last_y))

    zero_label_x = left + 8
    zero_label_y = min(max(zero_y - 7, top + 14), top + plot_h - 6)
    svg.append(f'<line x1="{left}" y1="{zero_y:.2f}" x2="{left+plot_w}" y2="{zero_y:.2f}" stroke="#111827" stroke-dasharray="6,5" stroke-width="1.75" opacity="0.95"/>')
    svg.append(f'<rect x="{zero_label_x-3:.2f}" y="{zero_label_y-13:.2f}" width="78" height="17" fill="white" opacity="0.86"/>')
    svg.append(f'<text x="{zero_label_x}" y="{zero_label_y:.2f}" text-anchor="start" font-size="12" font-weight="700" fill="#111827">0 reference</text>')

    panel_x = left + plot_w + 12
    legend_x = panel_x
    legend_y = top + 16
    legend_entries = [
        suspect_key
        for suspect_key in curves
        if len(curves[suspect_key][0]) and np.any(np.isfinite(curves[suspect_key][1]))
    ]
    svg.append(f'<text x="{legend_x}" y="{legend_y-16}" font-size="12" font-weight="700" fill="#111827">Suspect Models</text>')
    for idx, suspect_key in enumerate(legend_entries):
        y = legend_y + idx * 22
        color = COLORS.get(suspect_key, "#525252")
        label = model_label(suspect_key)
        svg.append(f'<line x1="{legend_x}" y1="{y}" x2="{legend_x+22}" y2="{y}" stroke="{color}" stroke-width="3"/>')
        svg.append(f'<text x="{legend_x+28}" y="{y+4}" font-size="11" fill="#111827">{label}</text>')

    final_x = panel_x
    final_y = legend_y + 22 * max(len(legend_entries), 1) + 24
    svg.append(f'<text x="{final_x}" y="{final_y}" font-size="12" font-weight="700" fill="#111827">Final Value</text>')
    for idx, (suspect_key, color, raw_last_y) in enumerate(final_value_rows):
        y = final_y + 18 + idx * 30
        svg.append(f'<text x="{final_x}" y="{y}" font-size="10.5" fill="{color}">{model_label(suspect_key)}:</text>')
        svg.append(f'<text x="{final_x}" y="{y+13}" font-size="10.5" fill="{color}">{raw_last_y:.4g}</text>')

    svg.append(f'<text x="{left + plot_w/2:.1f}" y="{height-16}" text-anchor="middle" font-size="13" fill="#374151">Total LLM calls</text>')
    svg.append(f'<text x="18" y="{top + plot_h/2:.1f}" transform="rotate(-90 18 {top + plot_h/2:.1f})" text-anchor="middle" font-size="13" fill="#374151">Estimator value</text>')
    svg.append('</svg>')
    return "\n".join(svg)


ASSET_DIR = PROJECT_ROOT / "assets"
ASSET_DIR.mkdir(exist_ok=True)


def _slug(text: str) -> str:
    return "_".join(
        "".join(ch.lower() if ch.isalnum() else " " for ch in text).split()
    )


def save_exact_svg_and_png(svg_text: str, output_stem: Path) -> None:
    """Save the exact displayed SVG and a PNG rasterization of that same SVG."""
    import re
    import subprocess
    import tempfile

    svg_path = output_stem.with_suffix('.svg')
    png_path = output_stem.with_suffix('.png')
    svg_path.write_text(svg_text, encoding='utf-8')

    size_match = re.search(r'<svg width="([0-9.]+)" height="([0-9.]+)"', svg_text)
    width = int(float(size_match.group(1))) if size_match else 930
    height = int(float(size_match.group(2))) if size_match else 440

    browser_candidates = [
        Path(r'C:\Program Files\Google\Chrome\Application\chrome.exe'),
        Path(r'C:\Program Files (x86)\Google\Chrome\Application\chrome.exe'),
        Path(r'C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe'),
        Path(r'C:\Program Files\Microsoft\Edge\Application\msedge.exe'),
    ]
    browser = next((candidate for candidate in browser_candidates if candidate.exists()), None)

    if browser is not None:
        html = f'''<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
html, body {{ margin: 0; padding: 0; width: {width}px; height: {height}px; overflow: hidden; background: white; }}
svg {{ display: block; }}
</style>
</head>
<body>{svg_text}</body>
</html>'''
        with tempfile.NamedTemporaryFile('w', suffix='.html', delete=False, encoding='utf-8') as handle:
            html_path = Path(handle.name)
            handle.write(html)
        try:
            subprocess.run(
                [
                    str(browser),
                    '--headless=new',
                    '--disable-gpu',
                    '--hide-scrollbars',
                    '--force-device-scale-factor=2',
                    f'--window-size={width},{height}',
                    f'--screenshot={png_path}',
                    html_path.as_uri(),
                ],
                check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
            )
            return
        finally:
            html_path.unlink(missing_ok=True)

    try:
        import cairosvg
    except ModuleNotFoundError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cairosvg'])
        import cairosvg

    cairosvg.svg2png(
        bytestring=svg_text.encode('utf-8'),
        write_to=str(png_path),
        scale=2.0,
    )


NOTEBOOK_EXPORT_PREFIX = "primary_7b_suspect_convergence"

html_parts = []
exported_pngs = []
for reference_key in REFERENCE_MODEL_KEYS:
    for method_name, estimator_fn in METHODS.items():
        curves = {
            suspect_key: parallel_estimator_curve(
                trial_results[(reference_key, suspect_key)], estimator_fn
            )
            for suspect_key in SUSPECT_MODEL_KEYS_BY_REFERENCE[reference_key]
        }
        svg_text = render_svg_plot(reference_key, method_name, curves)
        html_parts.append(svg_text)
        output_stem = ASSET_DIR / f"{NOTEBOOK_EXPORT_PREFIX}_reference_{reference_key}_{_slug(method_name)}"
        save_exact_svg_and_png(svg_text, output_stem)
        exported_pngs.append(output_stem.with_suffix('.png'))

display(HTML("<div>" + "<hr style='margin:24px 0;'>".join(html_parts) + "</div>"))
print("Saved exact SVG/PNG exports:")
for path in exported_pngs:
    print(path)


Saved exact SVG/PNG exports:
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\primary_7b_suspect_convergence_reference_14b_naive_method_i.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\primary_7b_suspect_convergence_reference_14b_naive_method_ii.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\primary_7b_suspect_convergence_reference_32b_naive_method_i.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\primary_7b_suspect_convergence_reference_32b_naive_method_ii.png
